In [1]:
!pip install optuna

In [3]:
import optuna
print(optuna.__version__)

ModuleNotFoundError: No module named 'optuna'

In [1]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# ==========================
# Load Dataset
# ==========================

df = pd.read_csv("car_MSRP.csv")

# Drop duplicates
df = df.drop_duplicates()

# Missing values
df["Engine HP"] = df["Engine HP"].fillna(df["Engine HP"].median())
df["Engine Cylinders"] = df["Engine Cylinders"].fillna(df["Engine Cylinders"].median())
df["Number of Doors"] = df["Number of Doors"].fillna(df["Number of Doors"].mode()[0])
df["Market Category"] = df["Market Category"].fillna("Unknown")
df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ==========================
# Features & Target
# ==========================

X = df.drop("MSRP", axis=1)
y = df["MSRP"]

# ==========================
# Train Test Split
# ==========================

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================
# Columns
# ==========================

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# ==========================
# Preprocessor
# ==========================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

# ==========================
# Cross Validation
# ==========================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ==========================
# Linear Regression
# ==========================

def objective_lr(trial):

    model = LinearRegression()

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

# ==========================
# Ridge
# ==========================

def objective_ridge(trial):

    model = Ridge(
        alpha=trial.suggest_float(
            "alpha",
            0.001,
            100,
            log=True
        )
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

# ==========================
# Lasso
# ==========================

def objective_lasso(trial):

    model = Lasso(
        alpha=trial.suggest_float(
            "alpha",
            0.0001,
            10,
            log=True
        )
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

# ==========================
# Decision Tree
# ==========================

def objective_dt(trial):

    model = DecisionTreeRegressor(
        max_depth=trial.suggest_int(
            "max_depth",
            2,
            30
        ),
        min_samples_split=trial.suggest_int(
            "min_samples_split",
            2,
            20
        ),
        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf",
            1,
            10
        ),
        random_state=42
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

# ==========================
# Random Forest
# ==========================

def objective_rf(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int(
            "n_estimators",
            100,
            500,
            step=50
        ),
        max_depth=trial.suggest_int(
            "max_depth",
            5,
            30
        ),
        min_samples_split=trial.suggest_int(
            "min_samples_split",
            2,
            20
        ),
        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf",
            1,
            10
        ),
        random_state=42,
        n_jobs=-1
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

# ==========================
# Gradient Boosting
# ==========================

def objective_gb(trial):

    model = GradientBoostingRegressor(
        n_estimators=trial.suggest_int(
            "n_estimators",
            100,
            500,
            step=50
        ),
        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),
        max_depth=trial.suggest_int(
            "max_depth",
            2,
            10
        ),
        random_state=42
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

# ==========================
# XGBoost
# ==========================

def objective_xgb(trial):

    model = XGBRegressor(
        n_estimators=trial.suggest_int(
            "n_estimators",
            100,
            500,
            step=50
        ),
        max_depth=trial.suggest_int(
            "max_depth",
            3,
            10
        ),
        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),
        subsample=trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),
        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),
        random_state=42
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    ).mean()

# ==========================
# Models Dictionary
# ==========================

objectives = {
    "LinearRegression": objective_lr,
    "Ridge": objective_ridge,
    "Lasso": objective_lasso,
    "DecisionTree": objective_dt,
    "RandomForest": objective_rf,
    "GradientBoosting": objective_gb,
    "XGBoost": objective_xgb
}

# ==========================
# Training
# ==========================

results = []

for model_name, obj_fn in objectives.items():

    print(f"\nOptimizing {model_name}")

    study = optuna.create_study(
        direction="maximize"
    )

    study.optimize(
        obj_fn,
        n_trials=10
    )

    results.append({
        "Model": model_name,
        "Best_R2": study.best_value,
        "Best_Params": study.best_params
    })

    print("Best R2 :", study.best_value)

# ==========================
# Results
# ==========================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Best_R2",
    ascending=False
)

print("\nFinal Ranking")
print(results_df)

# Best Model

best_model = results_df.iloc[0]

print("\nBest Model")
print(best_model)

KeyError: "['MSRP'] not found in axis"